# Dashboard inline (para testes dentro do notebook)

## Dados de reclamações do Nubank

In [12]:
from jupyter_dash import JupyterDash
from dash import dcc, html, Input, Output
from pathlib import Path
import plotly.express as px
import os
import sqlite3
import pandas as pd

# Conectar ao banco e carregar dados
BASE_DIR = Path.cwd()  # Pega a pasta do arquivo atual
ROOT_DIR = BASE_DIR.parent  # Sobe um nível (vai para Nubank)
db_path = ROOT_DIR / "data" / "nubank.db"   # Monta o caminho correto para o banco

conn = sqlite3.connect(db_path)
df_insights = pd.read_sql("SELECT * FROM reclamacoes_com_insights", conn)
conn.close()

# Criar app com JupyterDash
app = JupyterDash(__name__)

# Layout com banner roxo
app.layout = html.Div(
    style={"fontFamily": "Arial, sans-serif", "backgroundColor": "#f9f9f9", "padding": "20px"},
    children=[
        html.Div(
            children=html.H1(
                "Dashboard – Reclamações Nubank",
                style={
                    "textAlign": "center",
                    "color": "white",
                    "fontWeight": "bold",
                    "fontSize": "40px",
                    "margin": "0"
                }
            ),
            style={
                "backgroundColor": "#8A05BE",
                "padding": "20px"
            }
        ),

        dcc.Dropdown(
            id="categoria-dropdown",
            options=[{"label": cat, "value": cat} for cat in df_insights["Categoria_Primaria"].unique()],
            value=df_insights["Categoria_Primaria"].unique()[0],
            clearable=False,
            style={
                "width": "50%",
                "margin": "20px auto",
                "border": "2px solid #8A05BE",   # borda roxa Nubank
                "borderRadius": "8px",           # cantos arredondados
                "padding": "10px",
                "backgroundColor": "#f0e6f9",    # fundo lilás claro
                "color": "#8A05BE",              # texto roxo
                "fontWeight": "bold",
                "fontSize": "16px"
            }
        ),

        html.Div([
            dcc.Graph(id="grafico-barras", style={"display": "inline-block", "width": "49%"}),
            dcc.Graph(id="grafico-pizza", style={"display": "inline-block", "width": "49%"})
        ])
    ]
)

# Callback para atualizar gráficos
@app.callback(
    [Output("grafico-barras", "figure"),
     Output("grafico-pizza", "figure")],
    Input("categoria-dropdown", "value")
)
def update_graphs(categoria):
    df_filtrado = df_insights[df_insights["Categoria_Primaria"] == categoria]

    fig_bar = px.bar(
        df_filtrado,
        x="Severidade_Estimada",
        title=f"Severidade - {categoria}",
        color_discrete_sequence=["#8A05BE"]
    )

    fig_pie = px.pie(
        df_filtrado,
        names="Severidade_Estimada",
        title=f"Distribuição de Severidade - {categoria}",
        color_discrete_sequence=["#8A05BE", "#FF6F61", "#FFD700", "#00BFFF", "#32CD32"]
    )

    return fig_bar, fig_pie

# Rodar inline no notebook
app.run(mode="inline")

## Dados do Balancete do Nubank – 2021 a 2025

### Capital Principal

In [51]:
# Importar bibliotecas
import sqlite3
import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from jupyter_dash import JupyterDash
from dash import dcc, html

# Função utilitária para rodar queries
def run_query(sql):
    db_path = Path.cwd().parent / "data" / "nubank.db"
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query(sql, conn)
    conn.close()
    return df

# Query para pegar Capital Principal
df = run_query("""
    SELECT codigo, descricao, periodo, valor
    FROM historico_capital
    WHERE descricao = 'Capital Principal'
""")

# --- Normalização dos períodos ---
df["periodo"] = pd.to_datetime(df["periodo"], errors="coerce")
df = df.dropna(subset=["periodo"])
df["periodo_trim"] = df["periodo"].dt.to_period("Q").dt.end_time

# Consolidar por trimestre (último valor de cada trimestre)
df_trim = df.groupby("periodo_trim", as_index=False)["valor"].last()

# --- Forçar intervalo trimestral de 2021Q1 até 2025Q4 ---
periodos = pd.period_range("2021Q1", "2025Q4", freq="Q").to_timestamp("Q", "end")
df_periodos = pd.DataFrame({"periodo_trim": periodos})

# Merge para garantir todos os trimestres
df_trim = df_periodos.merge(df_trim, on="periodo_trim", how="left")

# Preencher valores ausentes com último valor conhecido
df_trim["valor"] = df_trim["valor"].ffill()

# Converter para bilhões
df_trim["valor_bilhoes"] = df_trim["valor"] / 1e9

# --- Criar app com estilo Nubank ---
app = JupyterDash(__name__)

app.layout = html.Div(
    style={
        "fontFamily": "Arial, sans-serif",
        "backgroundColor": "#f9f9f9",
        "padding": "20px"
    },
    children=[
        # Banner roxo
        html.Div(
            children=html.H1(
                "Dashboard – Capital Principal Nubank",
                style={
                    "textAlign": "center",
                    "color": "white",
                    "fontWeight": "bold",
                    "fontSize": "40px",
                    "margin": "0"
                }
            ),
            style={
                "backgroundColor": "#8A05BE",
                "padding": "20px"
            }
        ),

        # Gráficos lado a lado
        html.Div([
            dcc.Graph(
                id="grafico-linha",
                figure=px.line(
                    df_trim.sort_values("periodo_trim"),
                    x="periodo_trim",
                    y="valor_bilhoes",
                    title="Evolução do Capital Principal (2021–2025)",
                    markers=True,
                    labels={"periodo_trim": "Período", "valor_bilhoes": "Valor (B R$)"}
                ).update_layout(
                    plot_bgcolor="#ffffff",
                    paper_bgcolor="#f9f9f9",
                    font=dict(family="Arial", size=14, color="#333"),
                    title=dict(font=dict(size=20, color="#8A05BE")),
                    xaxis=dict(showgrid=True, gridcolor="#e0e0e0"),
                    yaxis=dict(showgrid=True, gridcolor="#e0e0e0", tickformat=".1f", ticksuffix="B")
                ),
                style={"display": "inline-block", "width": "49%"}
            ),

            dcc.Graph(
                id="grafico-barras",
                figure=px.bar(
                    df_trim.sort_values("periodo_trim"),
                    x="periodo_trim",
                    y="valor_bilhoes",
                    title="Capital Principal por Trimestre (2021–2025)",
                    labels={"periodo_trim": "Período", "valor_bilhoes": "Valor (B R$)"},
                    color_discrete_sequence=["#8A05BE"]
                ).update_layout(
                    plot_bgcolor="#ffffff",
                    paper_bgcolor="#f9f9f9",
                    font=dict(family="Arial", size=14, color="#333"),
                    title=dict(font=dict(size=20, color="#8A05BE")),
                    xaxis=dict(showgrid=True, gridcolor="#e0e0e0"),
                    yaxis=dict(showgrid=True, gridcolor="#e0e0e0", tickformat=".1f", ticksuffix="B")
                ),
                style={"display": "inline-block", "width": "49%"}
            )
        ])
    ]
)

app.run(mode="inline")

c:\Users\thacr\GitHub\Nubank\.venv\Lib\site-packages\jupyter_dash\jupyter_app.py:103: UserWarning: JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.
  super(JupyterDash, self).__init__(name=name, **kwargs)


### Evolução 2021-2025

In [61]:
# 1. Importar bibliotecas
import sqlite3
import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from jupyter_dash import JupyterDash
from dash import dcc, html, Input, Output
import dash

# 2. Função utilitária para rodar queries
def run_query(sql):
    db_path = Path.cwd().parent / "data" / "nubank.db"
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query(sql, conn)
    conn.close()
    return df

# 3. Carregar dados base
df_long = run_query("SELECT codigo, descricao, periodo, valor FROM historico_capital")
df_long["periodo"] = pd.to_datetime(df_long["periodo"], errors="coerce")
anos_disponiveis = sorted(df_long["periodo"].dropna().dt.year.unique())

# 4. Criar app
app = JupyterDash(__name__)

app.layout = html.Div(
    style={"fontFamily": "Arial, sans-serif", "backgroundColor": "#f9f9f9", "padding": "20px"},
    children=[
        # Banner
        html.Div(
            children=html.H1(
                "Dashboard – Indicadores de Capital Nubank",
                style={"textAlign": "center", "color": "white", "fontWeight": "bold", "fontSize": "40px", "margin": "0"}
            ),
            style={"backgroundColor": "#8A05BE", "padding": "20px"}
        ),

        # Botões de seleção de ano
        html.Div(
            id="botoes-anos",
            children=[
                *[
                    html.Button(str(ano), id=f"btn-{ano}", n_clicks=0,
                                style={"padding": "12px 28px", "margin": "10px",
                                       "backgroundColor": "#FFA500", "color": "white",
                                       "borderRadius": "10px", "border": "none",
                                       "cursor": "pointer", "fontWeight": "bold", "fontSize": "16px"})
                    for ano in anos_disponiveis
                ],
                html.Button("Evolução 2021–2025", id="btn-todos", n_clicks=0,
                            style={"padding": "12px 28px", "margin": "10px",
                                   "backgroundColor": "#250277", "color": "white",
                                   "borderRadius": "10px", "border": "none",
                                   "cursor": "pointer", "fontWeight": "bold", "fontSize": "16px"})
            ],
            style={"textAlign": "center", "marginBottom": "30px", "marginTop": "25px"}
        ),

        # Cards
        html.Div(id="cards", style={"display": "flex", "gap": "30px", "justifyContent": "center", "marginBottom": "30px"}),

        # Gráficos
        html.Div([
            dcc.Graph(id="grafico-capital", style={"display": "inline-block", "width": "49%"}),
            dcc.Graph(id="grafico-comp", style={"display": "inline-block", "width": "49%"})
        ]),
        html.Div([
            dcc.Graph(id="grafico-indices", style={"display": "inline-block", "width": "49%"}),
            dcc.Graph(id="grafico-margin", style={"display": "inline-block", "width": "49%"})
        ])
    ]
)

# 5. Callback para atualizar gráficos e cards
@app.callback(
    [Output("grafico-capital", "figure"),
     Output("grafico-comp", "figure"),
     Output("grafico-indices", "figure"),
     Output("grafico-margin", "figure"),
     Output("cards", "children")],
    [Input(f"btn-{ano}", "n_clicks") for ano in anos_disponiveis] + [Input("btn-todos", "n_clicks")],
    prevent_initial_call=False
)
def atualizar_graficos(*args):
    ctx = dash.callback_context
    if not ctx.triggered:
        ano, todos = 2025, False
    else:
        prop = ctx.triggered[0]["prop_id"].split(".")[0]
        todos, ano = (True, None) if prop == "btn-todos" else (False, int(prop.replace("btn-", "")))

    titulo_extra = "2021–2025" if todos else str(ano)

    # --- Caso especial: 2021 ---
    if ano == 2021 and not todos:
        df_capital = df_long[(df_long["descricao"] == "Capital Principal") & (df_long["periodo"].dt.year == 2021)] \
            .groupby("periodo", as_index=False)["valor"].last()

        fig_capital = px.line(df_capital, x="periodo", y="valor", markers=True,
                              title="Evolução do Capital Principal (2021)",
                              color_discrete_sequence=["#8A05BE"])

        # Apenas um card
        cards = []
        if not df_capital.empty:
            valor_capital = df_capital["valor"].iloc[-1] / 1e9
            cards.append(html.Div([
                html.H2("Capital Principal", style={"color": "#8A05BE", "margin": "0"}),
                html.H1(f"{valor_capital:.0f} B", style={"fontSize": "40px", "fontWeight": "bold", "color": "#333", "margin": "0"}),
                html.P(f"Último período: {df_capital['periodo'].iloc[-1].strftime('%Y-%m')}", style={"color": "#666", "margin": "0"})
            ], style={"backgroundColor": "white", "borderRadius": "12px", "padding": "20px",
                      "boxShadow": "0 4px 8px rgba(0,0,0,0.1)", "textAlign": "center", "width": "250px"}))

        # Retorna só o gráfico de capital e o card
        return fig_capital, go.Figure(), go.Figure(), go.Figure(), html.Div(cards, style={"display": "flex", "justifyContent": "center"})

    # --- Lógica normal para demais anos ou evolução ---
    if todos:
        df_capital = df_long[df_long["descricao"] == "Capital Principal"].groupby("periodo", as_index=False)["valor"].last()
        df_rwa = df_long[df_long["descricao"] == "RWA total"].groupby("periodo", as_index=False)["valor"].last()
        df_indices = df_long[df_long["descricao"].isin(["Índice de Capital Principal (ICP)", "Índice de Basileia"])] \
            .groupby(["descricao","periodo"], as_index=False)["valor"].last()
        df_margin = df_long[df_long["descricao"] == "Margem excedente de Capital Principal (%)"].groupby("periodo", as_index=False)["valor"].last()
    else:
        df_capital = df_long[(df_long["descricao"] == "Capital Principal") & (df_long["periodo"].dt.year == ano)] \
            .groupby("periodo", as_index=False)["valor"].last()
        df_rwa = df_long[(df_long["descricao"] == "RWA total") & (df_long["periodo"].dt.year == ano)] \
            .groupby("periodo", as_index=False)["valor"].last()
        df_indices = df_long[(df_long["descricao"].isin(["Índice de Capital Principal (ICP)", "Índice de Basileia"])) 
                             & (df_long["periodo"].dt.year == ano)] \
            .groupby(["descricao","periodo"], as_index=False)["valor"].last()
        df_margin = df_long[(df_long["descricao"] == "Margem excedente de Capital Principal (%)") 
                            & (df_long["periodo"].dt.year == ano)] \
            .groupby("periodo", as_index=False)["valor"].last()

    # Gráficos normais (4)
    fig_capital = px.line(df_capital, x="periodo", y="valor", markers=True,
                          title=f"Evolução do Capital Principal ({titulo_extra})",
                          color_discrete_sequence=["#8A05BE"])

    fig_comp = go.Figure()
    if not df_capital.empty:
        fig_comp.add_trace(go.Scatter(x=df_capital["periodo"], y=df_capital["valor"],
                                      mode="lines+markers", name="Capital Principal", line=dict(color="#8A05BE")))
    if not df_rwa.empty:
        fig_comp.add_trace(go.Scatter(x=df_rwa["periodo"], y=df_rwa["valor"],
                                      mode="lines+markers", name="RWA Total", line=dict(color="#FF6F61")))
    fig_comp.update_layout(title=f"Capital Principal vs RWA Total ({titulo_extra})")

    fig_indices = go.Figure()
    if not df_indices.empty:
        fig_indices = px.line(df_indices, x="periodo", y="valor", color="descricao", markers=True,
                              title=f"Índice de Capital Principal vs Índice de Basileia ({titulo_extra})",
                              color_discrete_map={"Índice de Capital Principal (ICP)": "#8A05BE",
                                                  "Índice de Basileia": "#FFD700"})

    fig_margin = px.line(df_margin, x="periodo", y="valor", markers=True,
                         title=f"Margem Excedente de Capital Principal (%) ({titulo_extra})",
                         color_discrete_sequence=["#32CD32"]) if not df_margin.empty else go.Figure()

    # Cards normais (3)
    cards = []
    if not df_capital.empty:
        valor_capital = df_capital["valor"].iloc[-1] / 1e9
        cards.append(html.Div([
            html.H2("Capital Principal", style={"color": "#8A05BE", "margin": "0"}),
            html.H1(f"{valor_capital:.0f} B", style={"fontSize": "40px", "fontWeight": "bold", "color": "#333", "margin": "0"})
        ], style={"backgroundColor": "white", "borderRadius": "12px", "padding": "20px",
                  "boxShadow": "0 4px 8px rgba(0,0,0,0.1)", "textAlign": "center", "width": "250px"}))

    if not df_rwa.empty:
        valor_rwa = df_rwa["valor"].iloc[-1] / 1e9
        cards.append(html.Div([
            html.H2("RWA Total", style={"color": "#FF6F61", "margin": "0"}),
            html.H1(f"{valor_rwa:.0f} B", style={"fontSize": "40px", "fontWeight": "bold", "color": "#333", "margin": "0"})
        ], style={"backgroundColor": "white", "borderRadius": "12px", "padding": "20px",
                  "boxShadow": "0 4px 8px rgba(0,0,0,0.1)", "textAlign": "center", "width": "250px"}))

    if not df_indices.empty:
        ultimo_periodo = df_indices["periodo"].max()
        valor_bas = df_indices[(df_indices["descricao"]=="Índice de Basileia") &
                               (df_indices["periodo"]==ultimo_periodo)]["valor"].values[0]
        cards.append(html.Div([
            html.H2("Índice de Basileia", style={"color": "#F2AC07", "margin": "0"}),
            html.H1(f"{valor_bas:.1f}%", style={"fontSize": "40px", "fontWeight": "bold", "color": "#333", "margin": "0"})
        ], style={"backgroundColor": "white", "borderRadius": "12px", "padding": "20px",
                  "boxShadow": "0 4px 8px rgba(0,0,0,0.1)", "textAlign": "center", "width": "250px"}))

    cards_layout = html.Div(cards, style={"display": "flex", "gap": "30px", "justifyContent": "center", "marginBottom": "30px"})

    return fig_capital, fig_comp, fig_indices, fig_margin, cards_layout

# 6. Rodar inline
app.run(mode="inline")

c:\Users\thacr\GitHub\Nubank\.venv\Lib\site-packages\jupyter_dash\jupyter_app.py:103: UserWarning: JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.
  super(JupyterDash, self).__init__(name=name, **kwargs)


In [ ]:
# 1. Importar bibliotecas
import sqlite3
import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from jupyter_dash import JupyterDash
from dash import dcc, html, Input, Output
import dash

# 2. Função utilitária para rodar queries
def run_query(sql):
    db_path = Path.cwd().parent / "data" / "nubank.db"
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query(sql, conn)
    conn.close()
    return df

# --- Preparar dados base ---
df_long = run_query("SELECT codigo, descricao, periodo, valor FROM historico_capital")
df_long["periodo"] = pd.to_datetime(df_long["periodo"], errors="coerce")
anos_disponiveis = sorted(df_long["periodo"].dropna().dt.year.unique())

# --- Criar app ---
app = JupyterDash(__name__)

app.layout = html.Div(
    style={"fontFamily": "Arial, sans-serif", "backgroundColor": "#f9f9f9", "padding": "20px"},
    children=[
        # Banner
        html.Div(
            children=html.H1(
                "Dashboard – Indicadores de Capital Nubank",
                style={
                    "textAlign": "center",
                    "color": "white",
                    "fontWeight": "bold",
                    "fontSize": "40px",
                    "margin": "0"
                }
            ),
            style={"backgroundColor": "#8A05BE", "padding": "20px"}
        ),
        
        # Linha com seleção de ano, botão evolução e cards
        html.Div([
            dcc.Dropdown(
                id="ano-dropdown",
                options=[{"label": str(ano), "value": ano} for ano in anos_disponiveis],
                value=2025,
                clearable=False,
                style={
                    "width": "100px",
                    "backgroundColor": "#8A05BE",
                    "color": "#8A05BE",
                    "fontWeight": "900",
                    "borderRadius": "8px",
                    "padding": "5px",
                    "marginLeft": "10px",
                    "marginRight": "25px"
                }
            ),
            html.Button("2021–2025", id="btn-todos", n_clicks=0,
                        style={
                            "padding": "12px 20px",
                            "marginRight": "40px",
                            "backgroundColor": "#FFA500",
                            "color": "white",
                            "borderRadius": "8px",
                            "border": "none",
                            "cursor": "pointer",
                            "fontWeight": "bold",
                            "fontSize": "16px"
                        }),
            html.Div(id="cards", style={"display": "flex", "gap": "20px", "marginTop": "40px","marginBottom": "30px", "marginLeft": "50px"})
        ], style={"display": "flex", "alignItems": "center", "marginBottom": "30px"}),

        # Gráficos
        html.Div([
            dcc.Graph(id="grafico-capital", style={"display": "inline-block", "width": "49%"}),
            dcc.Graph(id="grafico-comp", style={"display": "inline-block", "width": "49%"})
        ]),
        html.Div([
            dcc.Graph(id="grafico-indices", style={"display": "inline-block", "width": "49%"}),
            dcc.Graph(id="grafico-margin", style={"display": "inline-block", "width": "49%"})
        ])
    ]
)

@app.callback(
    [Output("grafico-capital", "figure"),
     Output("grafico-comp", "figure"),
     Output("grafico-indices", "figure"),
     Output("grafico-margin", "figure"),
     Output("cards", "children")],
    [Input("ano-dropdown", "value"),
     Input("btn-todos", "n_clicks")]
)
def atualizar_graficos(ano, n_todos):
    ctx = dash.callback_context
    todos = False
    if ctx.triggered and ctx.triggered[0]["prop_id"].split(".")[0] == "btn-todos":
        todos = True

    # --- Caso especial: 2021 ---
    if ano == 2021 and not todos:
        titulo_extra = "2021"
        df_capital = df_long[(df_long["descricao"] == "Capital Principal") & (df_long["periodo"].dt.year == 2021)] \
            .groupby("periodo", as_index=False)["valor"].last()

        fig_capital = px.line(df_capital, x="periodo", y="valor", markers=True,
                              title="Evolução do Capital Principal (2021)",
                              color_discrete_sequence=["#8A05BE"])

        # Apenas um card
        cards = []
        if not df_capital.empty:
            valor_capital = df_capital["valor"].iloc[-1] / 1e9
            cards.append(html.Div([
                html.H3("Capital Principal", style={"color": "#8A05BE", "margin": "0"}),
                html.H1(f"{valor_capital:.0f} B", style={"fontSize": "32px", "fontWeight": "bold", "margin": "0"})
            ], style={"backgroundColor": "white", "padding": "15px", "borderRadius": "10px",
                      "boxShadow": "0 4px 8px rgba(0,0,0,0.1)", "textAlign": "center", "width": "200px"}))

        return fig_capital, go.Figure(), go.Figure(), go.Figure(), html.Div(cards, style={"display": "flex", "justifyContent": "center"})

    # --- Lógica normal para demais anos ou evolução ---
    if todos:
        titulo_extra = "2021–2025"
        df_capital = df_long[df_long["descricao"] == "Capital Principal"].groupby("periodo", as_index=False)["valor"].last()
        df_rwa = df_long[df_long["descricao"] == "RWA total"].groupby("periodo", as_index=False)["valor"].last()
        df_indices = df_long[df_long["descricao"].isin(["Índice de Capital Principal (ICP)", "Índice de Basileia"])] \
            .groupby(["descricao","periodo"], as_index=False)["valor"].last()
        df_margin = df_long[df_long["descricao"] == "Margem excedente de Capital Principal (%)"].groupby("periodo", as_index=False)["valor"].last()
    else:
        titulo_extra = str(ano)
        df_capital = df_long[(df_long["descricao"] == "Capital Principal") & (df_long["periodo"].dt.year == ano)] \
            .groupby("periodo", as_index=False)["valor"].last()
        df_rwa = df_long[(df_long["descricao"] == "RWA total") & (df_long["periodo"].dt.year == ano)] \
            .groupby("periodo", as_index=False)["valor"].last()
        df_indices = df_long[(df_long["descricao"].isin(["Índice de Capital Principal (ICP)", "Índice de Basileia"])) 
                             & (df_long["periodo"].dt.year == ano)] \
            .groupby(["descricao","periodo"], as_index=False)["valor"].last()
        df_margin = df_long[(df_long["descricao"] == "Margem excedente de Capital Principal (%)") 
                            & (df_long["periodo"].dt.year == ano)] \
            .groupby("periodo", as_index=False)["valor"].last()

    # Gráficos normais
    fig_capital = px.line(df_capital, x="periodo", y="valor", markers=True,
                          title=f"Evolução do Capital Principal ({titulo_extra})",
                          color_discrete_sequence=["#8A05BE"])

    fig_comp = go.Figure()
    if not df_capital.empty:
        fig_comp.add_trace(go.Scatter(x=df_capital["periodo"], y=df_capital["valor"],
                                      mode="lines+markers", name="Capital Principal", line=dict(color="#8A05BE")))
    if not df_rwa.empty:
        fig_comp.add_trace(go.Scatter(x=df_rwa["periodo"], y=df_rwa["valor"],
                                      mode="lines+markers", name="RWA Total", line=dict(color="#FF6F61")))
    fig_comp.update_layout(title=f"Capital Principal vs RWA Total ({titulo_extra})")

    fig_indices = go.Figure()
    if not df_indices.empty:
        fig_indices = px.line(df_indices, x="periodo", y="valor", color="descricao", markers=True,
                              title=f"Índice de Capital Principal vs Índice de Basileia ({titulo_extra})",
                              color_discrete_map={"Índice de Capital Principal (ICP)": "#8A05BE",
                                                  "Índice de Basileia": "#FFD700"})

    fig_margin = px.line(df_margin, x="periodo", y="valor", markers=True,
                         title=f"Margem Excedente de Capital Principal (%) ({titulo_extra})",
                         color_discrete_sequence=["#32CD32"]) if not df_margin.empty else go.Figure()

    # Cards normais (3)
    cards = []
    if not df_capital.empty:
        valor_capital = df_capital["valor"].iloc[-1] / 1e9
        cards.append(html.Div([
            html.H3("Capital Principal", style={"color": "#8A05BE", "margin": "0"}),
            html.H1(f"{valor_capital:.0f} B", style={"fontSize": "32px", "fontWeight": "bold", "margin": "0"})
        ], style={"backgroundColor": "white", "padding": "15px", "borderRadius": "10px",
                  "boxShadow": "0 4px 8px rgba(0,0,0,0.1)", "textAlign": "center", "width": "200px"}))

    if not df_rwa.empty:
        valor_rwa = df_rwa["valor"].iloc[-1] / 1e9
        cards.append(html.Div([
            html.H3("RWA Total", style={"color": "#FF6F61", "margin": "0"}),
            html.H1(f"{valor_rwa:.0f} B", style={"fontSize": "32px", "fontWeight": "bold", "margin": "0"})
        ], style={"backgroundColor": "white", "padding": "15px", "borderRadius": "10px",
                  "boxShadow": "0 4px 8px rgba(0,0,0,0.1)", "textAlign": "center", "width": "200px"}))

    if not df_indices.empty:
        ultimo_periodo = df_indices["periodo"].max()
        valor_bas = df_indices[(df_indices["descricao"]=="Índice de Basileia") &
                               (df_indices["periodo"]==ultimo_periodo)]["valor"].values[0]
        cards.append(html.Div([
            html.H3("Índice de Basileia", style={"color": "#FFD700", "margin": "0"}),
            html.H1(f"{valor_bas:.1f}%", style={"fontSize": "32px", "fontWeight": "bold", "margin": "0"})
        ], style={"backgroundColor": "white", "padding": "15px", "borderRadius": "10px",
                  "boxShadow": "0 4px 8px rgba(0,0,0,0.1)", "textAlign": "center", "width": "200px"}))

    return fig_capital, fig_comp, fig_indices, fig_margin, cards

# Rodar inline
app.run(mode="inline")

c:\Users\thacr\GitHub\Nubank\.venv\Lib\site-packages\jupyter_dash\jupyter_app.py:103: UserWarning: JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.
  super(JupyterDash, self).__init__(name=name, **kwargs)


: 